<a href="https://colab.research.google.com/github/rionkuri22/Re-collect_Scottylabs/blob/main/recollect_vectorize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Test: Rion

In [18]:
!pip install -q pinecone google-generativeai

In [19]:
!pip install -q -U pinecone-client google-generativeai

Connect Pinecone

In [26]:
import json
from google.colab import userdata
from pinecone import Pinecone
import google.generativeai as genai

# Setup Auth
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))

# Connect to your manually created index
index_name = "recollect-test"
index = pc.Index(index_name)

# Quick Check: This should show 'dimension': 3072
print("Index Stats:", index.describe_index_stats())

Index Stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 24 Apr 2026 18:52:41 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '35',
                                    'x-pinecone-request-latency-ms': '35',
                                    'x-pinecone-response-duration-ms': '36'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


Vectorize

In [34]:
import json

json_path = '/content/rion_kurihara_resume.json' # Corrected path

with open(json_path, 'r') as f:
    user_data = json.load(f)

print("\n--- Sample Data Inspection ---")

def print_sample_item(section_name):
    section_data = user_data.get(section_name, [])
    if section_data:
        print(f"First item in '{section_name}' section:")
        print(json.dumps(section_data[0], indent=2))
    else:
        print(f"'{section_name}' section is empty or not found.")

print_sample_item('education')
print_sample_item('experience')
print_sample_item('projects')
print_sample_item('skills')


--- Sample Data Inspection ---
First item in 'education' section:
{
  "School Name": "Keio University",
  "Start Date": "Sep 2023",
  "End Date": "Sep 2027",
  "Notes": "",
  "Degree Name": "Bachelor of Arts in Policy Management",
  "Activities": "Kotosaka seminar (Business strategy), Watanabe seminar (American politics, democracy, and media)",
  "source": "linkedin"
}
First item in 'experience' section:
{
  "Company Name": "Carnegie Mellon University",
  "Title": "Research Assistant",
  "Description": "Spearheading research under Prof. Naama Ilany-Tzur on how AI chatbots can close the political perception gap through effective UI/UX design. Related: 67-368 UI/UX Research for Product Managers",
  "Location": "Pittsburgh, PA",
  "Started On": "Sep 2025",
  "Finished On": "Dec 2025",
  "source": "linkedin"
}
First item in 'projects' section:
{
  "name": "15113-hw2-crossy-road",
  "html_url": "https://github.com/rionkuri22/15113-hw2-crossy-road",
  "description": null,
  "language": "Pyt

In [30]:
import json

json_path = '/content/rion_kurihara_resume.json' # Assuming this is the correct path from previous context

with open(json_path, 'r') as f:
    resume_data = json.load(f)

print("Keys and their types in the resume JSON:")
for key, value in resume_data.items():
    value_type = type(value).__name__
    if isinstance(value, list):
        print(f"  - '{key}': {value_type} (length: {len(value)})")
    elif isinstance(value, dict):
        print(f"  - '{key}': {value_type} (keys: {list(value.keys())})")
    else:
        print(f"  - '{key}': {value_type}")

print("\nPlease review the output above. If there is a key representing a list of sections (e.g., 'experience', 'education'), we can use that for chunking.")

Keys and their types in the resume JSON:
  - 'personal_info': list (length: 3)
  - 'education': list (length: 4)
  - 'experience': list (length: 5)
  - 'projects': list (length: 23)
  - 'skills': list (length: 11)
  - 'awards_and_honors': list (length: 8)
  - 'languages': list (length: 6)
  - 'interests_and_thoughts': list (length: 6)
  - 'multimedia': list (length: 3)
  - 'raw_extracted_text': list (length: 1)

Please review the output above. If there is a key representing a list of sections (e.g., 'experience', 'education'), we can use that for chunking.


In [36]:
# --- Helper Function for 3072 Vectors ---
def get_embedding(text):
    # This model outputs 768 by default, but we can set the output_dimensionality
    # if the specific model version supports it, or use the 3072-native model.
    result = genai.embed_content(
        model="models/gemini-embedding-001", # Identified as available embedding model
        content=text,
        task_type="retrieval_document",
        output_dimensionality=3072 # Force matching your index size
    )
    return result['embedding']

# --- Load and Process ---
json_path = '/content/rion_kurihara_resume.json'

with open(json_path, 'r') as f:
    user_data = json.load(f)

# Get the person's name directly from the 'person_name' key
user_name = user_data.get('person_name', 'User')
vectors_to_upsert = []

# Process Education section
for i, edu in enumerate(user_data.get('education', [])):
    chunk_text = f"Education: {edu.get('School Name', '')} - {edu.get('Degree Name', '')}. Activities: {edu.get('Activities', '')}"
    vector = get_embedding(chunk_text)
    vectors_to_upsert.append({
        "id": f"{user_name.replace(' ', '_')}_education_{i}",
        "values": vector,
        "metadata": {
            "text": chunk_text,
            "source": "unified_json",
            "owner": user_name,
            "section": "education"
        }
    })

# Process Experience section
for i, exp in enumerate(user_data.get('experience', [])):
    chunk_text = f"Experience: {exp.get('Title', '')} at {exp.get('Company Name', '')}. Details: {exp.get('Description', '')}"
    vector = get_embedding(chunk_text)
    vectors_to_upsert.append({
        "id": f"{user_name.replace(' ', '_')}_experience_{i}",
        "values": vector,
        "metadata": {
            "text": chunk_text,
            "source": "unified_json",
            "owner": user_name,
            "section": "experience"
        }
    })

# Process Projects section
for i, proj in enumerate(user_data.get('projects', [])):
    chunk_text = f"Project: {proj.get('name', '')}. Details: {proj.get('description', '') if proj.get('description') else ''}"
    vector = get_embedding(chunk_text)
    vectors_to_upsert.append({
        "id": f"{user_name.replace(' ', '_')}_project_{i}",
        "values": vector,
        "metadata": {
            "text": chunk_text,
            "source": "unified_json",
            "owner": user_name,
            "section": "project"
        }
    })

# Process Skills section (as a single chunk for all skills)
skills_list = []
for skill_entry in user_data.get('skills', []):
    if isinstance(skill_entry, dict) and 'Name' in skill_entry:
        skills_list.append(skill_entry['Name'])
    elif isinstance(skill_entry, str):
        skills_list.append(skill_entry)

if skills_list:
    skills_text = "Skills: " + ", ".join(skills_list)
    vector = get_embedding(skills_text)
    vectors_to_upsert.append({
        "id": f"{user_name.replace(' ', '_')}_skills_0",
        "values": vector,
        "metadata": {
            "text": skills_text,
            "source": "unified_json",
            "owner": user_name,
            "section": "skills"
        }
    })

# --- Push to Pinecone ---
if vectors_to_upsert:
    index.upsert(vectors=vectors_to_upsert)
    print(f"Successfully vectorized and pushed {len(vectors_to_upsert)} chunks to Pinecone.")
else:
    print("No vectors were generated. Please check your JSON data for relevant sections (education, experience, projects, skills).")

Successfully vectorized and pushed 33 chunks to Pinecone.


Test chat

In [ ]:
def ask_recollect(question):
    # 1. Turn the question into a 3072 vector
    query_vector = get_embedding(question) # Uses your existing function

    # 2. Get relevant snippets from Pinecone
    results = index.query(
        vector=query_vector,
        top_k=5,
        include_metadata=True
    )

    # 3. Combine snippets into one "Context" block
    context_text = "\n---\n".join([match['metadata']['text'] for match in results['matches']])

    # 4. Create the "System Instruction" for Gemini
    prompt = f"""
    You are Re:collect, a smart professional assistant.
    Use the following verified data about Rion Kurihara to answer the user's question.
    If the answer isn't in the context, say you don't know based on the current records.

    CONTEXT:
    {context_text}

    USER QUESTION: {question}
    """

    # 5. Generate Answer
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)

    return response.text

# --- TRY IT OUT ---
print(ask_recollect("What is Rion's experience with AI and research?"))